# WAL ETL Analytics

Анализ скорости загрузки RAW → DDS через PostgreSQL WAL

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from datetime import datetime, timedelta
import glob
import os

## 1. Загрузка данных

In [ ]:
# Find latest WAL history file
wal_files = glob.glob('../data/wal_history_*.csv')
if wal_files:
    latest_wal = max(wal_files)
    print(f'Loading: {latest_wal}')
    wal_df = pd.read_csv(latest_wal, parse_dates=['timestamp'])
    print(f'Loaded {len(wal_df)} records')
else:
    print('No WAL history files found')
    wal_df = pd.DataFrame()

In [ ]:
# Find latest activity history file
activity_files = glob.glob('../data/activity_history_*.csv')
if activity_files:
    latest_activity = max(activity_files)
    print(f'Loading: {latest_activity}')
    activity_df = pd.read_csv(latest_activity, parse_dates=['timestamp'])
    print(f'Loaded {len(activity_df)} records')
else:
    print('No activity history files found')
    activity_df = pd.DataFrame()

## 2. Анализ WAL

In [ ]:
if not wal_df.empty:
    print('WAL Statistics:')
    print(f'  Start: {wal_df["timestamp"].min()}')
    print(f'  End: {wal_df["timestamp"].max()}')
    print(f'  Duration: {wal_df["timestamp"].max() - wal_df["timestamp"].min()}')
    print(f'  Min WAL: {wal_df["wal_gb"].min():.2f} GB')
    print(f'  Max WAL: {wal_df["wal_gb"].max():.2f} GB')
    print(f'  Total growth: {wal_df["wal_gb"].max() - wal_df["wal_gb"].min():.2f} GB')
    print(f'  Avg speed: {wal_df["wal_speed_mb_min"].mean():.0f} MB/min')
    print(f'  Max speed: {wal_df["wal_speed_mb_min"].max():.0f} MB/min')

In [ ]:
# WAL Growth Chart
if not wal_df.empty:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # WAL Growth
    axes[0].plot(wal_df['timestamp'], wal_df['wal_gb'], 'b-', linewidth=2, label='WAL Size')
    axes[0].fill_between(wal_df['timestamp'], wal_df['wal_gb'], alpha=0.3)
    axes[0].set_title('WAL Growth', fontsize=14)
    axes[0].set_ylabel('WAL (GB)')
    axes[0].grid(True, alpha=0.3)
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    
    # WAL Speed
    axes[1].plot(wal_df['timestamp'], wal_df['wal_speed_mb_min'], 'r-', linewidth=2, label='Speed')
    axes[1].axhline(y=wal_df['wal_speed_mb_min'].mean(), color='g', linestyle='--', label='Average')
    axes[1].set_title('WAL Speed', fontsize=14)
    axes[1].set_ylabel('Speed (MB/min)')
    axes[1].set_xlabel('Time')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    
    plt.tight_layout()
    plt.savefig('../output/wal_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3. Анализ активности

In [ ]:
if not activity_df.empty:
    # Filter to specific query if needed
    target_activity = activity_df[activity_df['query'].str.contains('alk_markserial', na=False)]
    
    if not target_activity.empty:
        print('Target Activity (alk_markserial):')
        print(f'  Records: {len(target_activity)}')
        print(f'  First seen: {target_activity["timestamp"].min()}')
        print(f'  Last seen: {target_activity["timestamp"].max()}')
        print(f'  Duration: {target_activity["timestamp"].max() - target_activity["timestamp"].min()}')
        
        # Get unique PIDs
        pids = target_activity['pid'].unique()
        print(f'  PIDs: {pids}')
        
        # Get wait events
        wait_events = target_activity['wait_event'].value_counts()
        print(f'\nWait Events:')
        for event, count in wait_events.items():
            print(f'  {event}: {count}')

In [ ]:
# Active queries timeline
if not activity_df.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Count active queries per timestamp
    activity_counts = activity_df.groupby('timestamp').size()
    
    ax.plot(activity_counts.index, activity_counts.values, 'b-', linewidth=2)
    ax.fill_between(activity_counts.index, activity_counts.values, alpha=0.3)
    ax.set_title('Active Queries Over Time', fontsize=14)
    ax.set_ylabel('Number of Active Queries')
    ax.set_xlabel('Time')
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    
    plt.tight_layout()
    plt.savefig('../output/activity_timeline.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Прогноз завершения

In [ ]:
def estimate_completion(wal_df, target_wal_gb=None):
    """Estimate time to complete ETL operation."""
    if wal_df.empty or len(wal_df) < 2:
        return None
    
    current = wal_df.iloc[-1]
    start = wal_df.iloc[0]
    
    # Calculate stats
    elapsed_minutes = (current['timestamp'] - start['timestamp']).total_seconds() / 60
    wal_generated = current['wal_gb'] - start['wal_gb']
    avg_speed = current['wal_speed_mb_min']
    
    print('=' * 60)
    print('ETL OPERATION ANALYSIS')
    print('=' * 60)
    print(f'Started: {start["timestamp"]}')
    print(f'Runtime: {elapsed_minutes:.0f} minutes ({elapsed_minutes/60:.1f} hours)')
    print(f'Current WAL: {current["wal_gb"]:.2f} GB')
    print(f'WAL Generated: {wal_generated:.2f} GB')
    print(f'Average Speed: {avg_speed:.0f} MB/min')
    
    if target_wal_gb:
        remaining_gb = target_wal_gb - current['wal_gb']
        if remaining_gb > 0 and avg_speed > 0:
            remaining_minutes = (remaining_gb * 1024) / avg_speed
            finish_time = current['timestamp'] + timedelta(minutes=remaining_minutes)
            print(f'\nTarget: {target_wal_gb} GB')
            print(f'Remaining: {remaining_gb:.2f} GB')
            print(f'ETA: {finish_time}')
    
    print('=' * 60)
    
    return {
        'elapsed_minutes': elapsed_minutes,
        'wal_generated': wal_generated,
        'avg_speed': avg_speed
    }

In [ ]:
# Run analysis
if not wal_df.empty:
    stats = estimate_completion(wal_df, target_wal_gb=200)

## 5. Анализ узких мест (Bottleneck)

In [ ]:
def analyze_bottlenecks(activity_df):
    """Analyze potential bottlenecks from wait events."""
    if activity_df.empty:
        return
    
    print('=' * 60)
    print('BOTTLENECK ANALYSIS')
    print('=' * 60)
    
    # Analyze wait events
    wait_events = activity_df[activity_df['wait_event'].notna()]['wait_event'].value_counts()
    
    if not wait_events.empty:
        print('\nWait Events Distribution:')
        for event, count in wait_events.head(10).items():
            print(f'  {event}: {count}')
        
        # Identify bottleneck
        top_event = wait_events.index[0]
        print(f'\nDetected Bottleneck: {top_event}')
        
        # Recommendations
        recommendations = {
            'WALWriteLock': 'Reduce transaction size or use batch update',
            'Lock': 'Check for deadlocks or long-running transactions',
            'IO': 'Check disk I/O or use faster storage',
            'CPU': 'Consider parallel processing or indexing',
            'BufferContent': 'Increase shared_buffers',
            'Activity': 'Check for lock contention'
        }
        
        for key, rec in recommendations.items():
            if key.lower() in top_event.lower():
                print(f'Recommendation: {rec}')
                break
    else:
        print('No wait events recorded')
    
    print('=' * 60)

In [ ]:
# Run bottleneck analysis
if not activity_df.empty:
    analyze_bottlenecks(activity_df)

## 6. RAW → DDS Comparison

In [ ]:
def compare_raw_dds(wal_df, raw_rows=None, dds_rows=None):
    """Compare RAW and DDS metrics."""
    if wal_df.empty:
        return
    
    print('=' * 60)
    print('RAW → DDS COMPARISON')
    print('=' * 60)
    
    current = wal_df.iloc[-1]
    start = wal_df.iloc[0]
    
    wal_generated = current['wal_gb'] - start['wal_gb']
    elapsed_minutes = (current['timestamp'] - start['timestamp']).total_seconds() / 60
    
    print(f'\nWAL Generated: {wal_generated:.2f} GB')
    print(f'Runtime: {elapsed_minutes:.0f} minutes')
    
    if raw_rows:
        print(f'\nRAW Rows: {raw_rows:,}')
        print(f'WAL per million rows: {(wal_generated / raw_rows * 1_000_000):.2f} GB')
        print(f'Rows per minute: {raw_rows / elapsed_minutes:,.0f}')
    
    if dds_rows:
        print(f'\nDDS Rows: {dds_rows:,}')
        print(f'Rows per minute: {dds_rows / elapsed_minutes:,.0f}')
    
    print('=' * 60)

In [ ]:
# Run comparison (update with actual row counts)
if not wal_df.empty:
    compare_raw_dds(wal_df, raw_rows=151_817_640)

## 7. Экспорт отчёта

In [ ]:
def export_html_report(wal_df, activity_df, output_dir='../output'):
    """Export analysis to HTML report."""
    os.makedirs(output_dir, exist_ok=True)
    
    date_str = datetime.now().strftime('%Y%m%d')
    report_file = f'{output_dir}/WAL_Report_{date_str}.html'
    
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>WAL ETL Report - {date_str}</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; }}
            h1 {{ color: #333; }}
            .metric {{ background: #f5f5f5; padding: 10px; margin: 10px 0; border-radius: 5px; }}
            .metric-label {{ font-weight: bold; color: #666; }}
            .metric-value {{ font-size: 1.2em; color: #333; }}
        </style>
    </head>
    <body>
        <h1>WAL ETL Performance Report</h1>
        <p>Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
    """
    
    if not wal_df.empty:
        current = wal_df.iloc[-1]
        start = wal_df.iloc[0]
        
        html_content += f"""
        <h2>WAL Statistics</h2>
        <div class="metric">
            <div class="metric-label">Current WAL</div>
            <div class="metric-value">{current['wal_gb']:.2f} GB</div>
        </div>
        <div class="metric">
            <div class="metric-label">WAL Generated</div>
            <div class="metric-value">{current['wal_gb'] - start['wal_gb']:.2f} GB</div>
        </div>
        <div class="metric">
            <div class="metric-label">Average Speed</div>
            <div class="metric-value">{wal_df['wal_speed_mb_min'].mean():.0f} MB/min</div>
        </div>
        """
    
    html_content += """
    </body>
    </html>
    """
    
    with open(report_file, 'w') as f:
        f.write(html_content)
    
    print(f'Report saved to: {report_file}')
    return report_file

In [ ]:
# Export report
if not wal_df.empty:
    report_file = export_html_report(wal_df, activity_df)